# RAPTOR on QASPER

End-to-end demo: load a QASPER document → build RAPTOR tree → generate summary → evaluate against abstract.

**Architecture** (ICLR 2024 paper, Section 3):
1. Text extraction from QASPER structured format
2. Chunking + SBERT embedding (leaf nodes)
3. Recursive cluster → summarize → embed (tree construction)
4. Collapsed tree retrieval
5. Final summary generation + ROUGE/BERTScore evaluation

## Section 0 — Setup

In [ ]:
import os, sys
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env

# Make both raptor/ and the project root importable
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('raptor'))

print('OPENAI_API_KEY set:', bool(os.environ.get('OPENAI_API_KEY')))

## Section 1 — Load QASPER document

QASPER paper: *Dasigi et al., 2021*. Each entry has `title`, `abstract`, and `full_text` (section_name / paragraphs).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import logging
logging.disable(logging.CRITICAL)

import os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'

from datasets import load_dataset

dataset = load_dataset('allenai/qasper', split='validation')
document = dataset[0]

logging.disable(logging.NOTSET)

print(f'Title   : {document["title"]}')
print(f'Sections: {document["full_text"]["section_name"][:5]}')

### Verify: text extraction

In [ ]:
import importlib, qasper_pipeline
importlib.reload(qasper_pipeline)
from qasper_pipeline import extract_qasper_text

text = extract_qasper_text(document)
words = text.split()
print(f'Total words : {len(words)}')
print(f'Preview     : {text[:300]}...')

## Section 2 — Configure RAPTOR

Paper Appendix D: summarization uses GPT with the prompt *"Write a summary of the following, including as many key details as possible"*.  
We use **gpt-4.1-nano** (cost-efficient) + **SBERT multi-qa-mpnet-base-cos-v1** (768-dim, L2-normalised) for embeddings.

In [ ]:
importlib.reload(qasper_pipeline)
from qasper_pipeline import build_raptor_config

config = build_raptor_config()
print('Config ready.')
print(config.log_config())

## Section 3 — Build RAPTOR tree

Paper Section 3: recursive embed → cluster (UMAP + GMM) → summarize until ≤1 cluster or ≤ min cluster size.  
Uses **collapsed tree** retrieval (Section 3.2) — proven better than tree traversal.

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('raptor'))

from raptor import RetrievalAugmentation

ra = RetrievalAugmentation(config=config)
ra.add_documents(text)
print('Tree built.')
print(f'Tree layers : {len(ra.tree.layer_to_nodes)}')
print(f'Total nodes : {sum(len(v) for v in ra.tree.layer_to_nodes.values())}')

## Section 4 — Generate summary

Retrieve broad context from the collapsed tree (top_k=10, max_tokens=3500), then call gpt-4.1-nano to produce the final summary.

In [ ]:
from openai import OpenAI

context, layer_info = ra.retrieve(
    question='Summarize this document',
    top_k=10,
    max_tokens=3500,
    collapse_tree=True,
    return_layer_information=True,
)
print(f'Retrieved context length: {len(context.split())} words')
print(f'Layer info: {layer_info}')

In [ ]:
client = OpenAI()
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role': 'system', 'content': 'You are a document summarization assistant.'},
        {
            'role': 'user',
            'content': (
                'Write a concise but complete summary of the document based on '
                f'the following content:\n\n{context}'
            ),
        },
    ],
    max_tokens=500,
)
summary = response.choices[0].message.content.strip()
print('SUMMARY:')
print(summary)

## Section 5 — Evaluate

Score the generated summary against the QASPER paper abstract using ROUGE-1/2/L and BERTScore (roberta-large).

In [ ]:
importlib.reload(qasper_pipeline)
from qasper_pipeline import print_evaluation

reference = document['abstract']
scores = print_evaluation(summary, reference)

## Section 6 — Batch evaluation (optional)

Run the full pipeline over N QASPER documents and report average scores.

In [ ]:
importlib.reload(qasper_pipeline)
from qasper_pipeline import summarize_document, evaluate_summary
import numpy as np

N = 3  # increase as needed
all_scores = []

for i in range(N):
    doc = dataset[i]
    print(f'\n[{i+1}/{N}] {doc["title"][:60]}')
    result = summarize_document(doc)  # builds fresh tree per doc
    s = evaluate_summary(result['summary'], result['abstract'])
    all_scores.append(s)
    print(f'  ROUGE-L={s["rougeL"]:.3f}  BERTScore-F1={s["bertscore_f1"]:.3f}')

print('\n--- AVERAGE ---')
for k in all_scores[0]:
    print(f'  {k}: {np.mean([s[k] for s in all_scores]):.4f}')